# Merge Top + Secondary + Tertiary Extractions

Combines the three per-tier merged CSVs (already HTML+text merged) into one
flat dataset keyed by normalized `document_id`.

**Merge order:**
1. Load `merged_top.csv`, `merged_secondary.csv`, `merged_tertiary.csv`
2. Normalize doc IDs (remove spaces/dots/dashes, uppercase)
3. Outer-join all three on `document_id`
4. Save as `output/merged_all.csv`

*(PDF AcroForm merge is in Section 5 — run separately once ready.)*

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from merge.merge_extractions import normalize_doc_id, is_valid_waiver_id, is_empty, compute_fill_rate

import pandas as pd
import numpy as np

In [ ]:
# ── Configuration ──
TOP_CSV       = '../output/merged_top.csv'
SECONDARY_CSV = '../output/merged_secondary.csv'
TERTIARY_CSV  = '../output/merged_tertiary.csv'
OUTPUT_CSV    = '../output/merged_all.csv'

PDF_CSV       = '../output/pdf_acroform_extraction.csv'  # used in Section 5

## 1. Load tier CSVs

In [ ]:
df_top  = pd.read_csv(TOP_CSV)
df_sec  = pd.read_csv(SECONDARY_CSV)
df_ter  = pd.read_csv(TERTIARY_CSV)

print(f'Top:       {df_top.shape}')
print(f'Secondary: {df_sec.shape}')
print(f'Tertiary:  {df_ter.shape}')

## 2. Normalize doc IDs and filter invalid rows

In [ ]:
def prep(df, label):
    df = df.copy()
    df['document_id'] = df['document_id'].apply(normalize_doc_id)
    before = len(df)
    df = df[df['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
    df = df.drop_duplicates(subset=['document_id'], keep='first')
    print(f'{label}: {before} → {len(df)} rows after filtering/dedup')
    return df

df_top = prep(df_top, 'Top')
df_sec = prep(df_sec, 'Secondary')
df_ter = prep(df_ter, 'Tertiary')

## 3. Outer-join all three tiers on document_id

In [ ]:
merged = (
    df_top
    .merge(df_sec, on='document_id', how='outer', suffixes=('', '_sec'))
    .merge(df_ter, on='document_id', how='outer', suffixes=('', '_ter'))
)

# Replace NaN strings with empty string for consistency
merged = merged.fillna('')

print(f'Merged shape: {merged.shape}')
print(f'Unique doc IDs: {merged["document_id"].nunique()}')
display(merged.head(5))

## 4. Save merged_all.csv

In [ ]:
import os, csv

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved {len(merged)} records → {OUTPUT_CSV}')

## 5. Merge with PDF AcroForm

Outer-join `pdf_acroform_extraction.csv` onto `merged_all` by `document_id`
so that docs present only in the PDF (43 of them) are also kept.
PDF columns are added as-is — radio/checkbox values read directly from the form.
Final columns are reordered into the agreed variable order.

In [ ]:
import os, csv

PDF_FINAL_CSV = '../output/merged_all_with_pdf.csv'

df_pdf = pd.read_csv(PDF_CSV)
df_pdf['document_id'] = df_pdf['document_id'].apply(normalize_doc_id)
df_pdf = df_pdf[df_pdf['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
df_pdf = df_pdf.drop_duplicates(subset=['document_id'], keep='first')
print(f'PDF AcroForm: {df_pdf.shape}')

# Outer join — keeps docs that appear only in PDF too
merged_with_pdf = merged.merge(df_pdf, on='document_id', how='outer')
merged_with_pdf = merged_with_pdf.fillna('')

ids_only_pdf = set(df_pdf['document_id']) - set(merged['document_id'])
print(f'Only in top+sec+ter: {len(set(merged["document_id"]) - set(df_pdf["document_id"]))}')
print(f'Only in pdf_acroform: {len(ids_only_pdf)}')
print(f'Overlap: {len(set(merged["document_id"]) & set(df_pdf["document_id"]))}')
print(f'Total rows after join: {len(merged_with_pdf)}')

# ── Column order ──────────────────────────────────────────────────────────────
FINAL_COLUMN_ORDER = [
    # Key
    'document_id',

    # Waiver overview
    'title', 'waiver_type', 'effective_date', 'approval_period',

    # Level of care
    'hospital_loc', 'hospital_loc_limits',
    'nursing_facility_loc', 'nursing_facility_loc_limits',
    'ifc_loc', 'ifc_loc_limits',
    'local_eval', 'local_eval_instrument', 'reeval_sched',

    # Concurrent waivers
    'concurrent_1915a', 'concurrent_1915b', 'concurrent_1932a',
    'concurrent_1915i', 'concurrent_1915j', 'concurrent_1115',

    # Eligibility — geography & groups
    'dual_elg', 'waive_geographic_limits', 'waive_geographic_lipd',
    'aged_group', 'aged_group_min', 'aged_group_max',
    'physicaldis_group', 'physicaldis_group_min', 'physicaldis_group_max',
    'otherdis_group', 'otherdis_group_min', 'otherdis_group_max',
    'braininjury_group', 'braininjury_group_min', 'braininjury_group_max',
    'hivaids_group', 'hivaids_group_min', 'hivaids_group_max',
    'medicallyfrail_group', 'medicallyfrail_group_min', 'medicallyfrail_group_max',
    'techdep_group', 'techdep_group_min', 'techdep_group_max',
    'autism_group', 'autism_group_min', 'autism_group_max',
    'dd_group', 'dd_group_min', 'dd_group_max',
    'id_group', 'id_group_min', 'id_group_max',
    'mi_group', 'mi_group_min', 'mi_group_max',
    'sed_group', 'sed_group_min', 'sed_group_max',

    # Eligibility — criteria
    'eligibility_1', 'eligibility_2', 'eligibility_3', 'eligibility_4',
    'eligibility_5', 'eligibility_5_percent', 'eligibility_5_100',
    'eligibility_6', 'eligibility_7', 'eligibility_8',
    'eligibility_9', 'eligibility_10', 'eligibility_11', 'eligibility_12',
    'spousal_impov_a', 'spousal_impov_bc',

    # Cost / enrollment
    'cost_limit_pcntaboveinstit', 'costlimit',
    'numberofbenes_year1', 'numberofbenes_year2', 'numberofbenes_year3',
    'numberofbenes_year4', 'numberofbenes_year5',
    'max_numberofbenes_year1', 'max_numberofbenes_year2', 'max_numberofbenes_year3',
    'max_numberofbenes_year4', 'max_numberofbenes_year5',
    'numberbenes_limited', 'phaseinoutschedule', 'entrantselection',
    'specialHCBS', 'enhanced_payments_yes', 'statecontracts_mcos',

    # Self-direction (Appendix E)
    'selfdirection_yes', 'selfdirection_description',
    'sd_authority', 'sd_election',
    'sd_livarrngmnt_1', 'sd_livarrngmnt_2', 'sd_livarrngmnt_3',
    'sd_service_1', 'sd_service_1_ea', 'sd_service_1_ba',
    'sd_fms_gov', 'sd_fms_pe',
    'scope_fms_1', 'scope_fms_2', 'scope_fms_3', 'scope_fms_4',
    'sd_numenrollees_ea1', 'sd_numenrollees_ea2', 'sd_numenrollees_ea3',
    'sd_numenrollees_ea4', 'sd_numenrollees_ea5',
    'sd_numenrollees_ba1', 'sd_numenrollees_ba2', 'sd_numenrollees_ba3',
    'sd_numenrollees_ba4', 'sd_numenrollees_ba5',
    'sd_coemployer', 'sd_commonlaw',
    'provider_rate_methods',
    'payforresidential', 'reimburse_paidcg',

    # Waiver services (Tertiary)
    'ma_1', 'ma_2', 'ma_3', 'ma_4', 'ma_5', 'ma_6',
    'ma_7', 'ma_8', 'ma_9', 'ma_10', 'ma_11', 'ma_12',
    'osa_1', 'osa_2', 'osa_3', 'osa_4', 'osa_5', 'osa_6',
    'osa_7', 'osa_8', 'osa_9', 'osa_10', 'osa_11', 'osa_12',
    'ce_1', 'ce_2', 'ce_3', 'ce_4', 'ce_5', 'ce_6',
    'ce_7', 'ce_8', 'ce_9', 'ce_10', 'ce_11', 'ce_12',
    'inse_1', 'inse_2', 'inse_3', 'inse_4', 'inse_5', 'inse_6',
    'inse_7', 'inse_8', 'inse_9', 'inse_10', 'inse_11', 'inse_12',

    # Descriptions / Transition
    'waiver_description',
    'transition_plan_1', 'transition_plan_2', 'transition_plan_3',
    'transition_plan_4', 'transition_plan_5', 'transition_plan_6',
    'transition_plan_7', 'transition_plan_8', 'transition_plan_9',
    'transition_plan_10',
]

final_cols = [c for c in FINAL_COLUMN_ORDER if c in merged_with_pdf.columns]
missing = [c for c in FINAL_COLUMN_ORDER if c not in merged_with_pdf.columns]
if missing:
    print(f'Warning — columns in order list but not in data: {missing}')

merged_with_pdf = merged_with_pdf[final_cols]

print(f'Final shape: {merged_with_pdf.shape}')
merged_with_pdf.to_csv(PDF_FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved {len(merged_with_pdf)} records → {PDF_FINAL_CSV}')

display(merged_with_pdf.head(3))